<a href="https://colab.research.google.com/github/ClaudioSeveroo/ciencia_de_dados/blob/main/projeto_analise_cenipa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ocorrências aeronáuticas no Brasil (2007 a 2026)
### Análise dos dados do CENIPA usando pandas

**Aluno:** _[Cláudio Roberto Severo Filho]_  |  **Disciplina:** _[Programação Para Ciência de Dados]_  |  **Dataset:** `ocorrencia.csv`

**Fonte dos dados:** CENIPA (Centro de Investigação e Prevenção de Acidentes Aeronáuticos), publicado no Portal Brasileiro de Dados Abertos:
https://dados.gov.br/dados/conjuntos-dados/ocorrencias-aeronauticas-da-aviacao-civil-brasileira

## 1. Definição do problema

**Tema:** ocorrências aeronáuticas (acidentes e incidentes) da aviação civil brasileira, notificadas ao CENIPA.

**Problema:** como as ocorrências aeronáuticas se distribuem no Brasil ao longo do tempo, entre os estados e ao longo do dia, e o que diferencia os acidentes, os incidentes graves e os incidentes?

**O que quero descobrir com os dados:**
1. Quantas ocorrências existem de cada classificação (acidente, incidente grave, incidente).
2. Como o número de ocorrências mudou de um ano para outro.
3. Quais estados, cidades e aeródromos concentram mais ocorrências.
4. Em que horários do dia elas acontecem mais.
5. Com que frequência há saída de pista em cada classificação.
6. Como está a investigação: quantas já terminaram, quantos relatórios foram publicados e quantas recomendações de segurança foram geradas.

**Por que esse dataset?** É um dado real e público, com mais de 14 mil registros, com texto, datas, horas e categorias, e alguns problemas de dados reais para tratar (marcadores como `***`, colunas repetidas, valores ausentes). O tema também é interessante e fácil de explicar.

**Resumo simplificado das classificações:**
* **Acidente:** ocorrência mais grave (por exemplo, com morte, lesão grave ou dano importante na aeronave).
* **Incidente grave:** quase foi um acidente.
* **Incidente:** ocorrência menos grave, mas que afeta ou poderia afetar a segurança.

## 2. Obtenção e exploração dos dados

### 2.1 Importando o arquivo
Irei utilizar o Pandas. O arquivo separa as colunas com `;` e usa o encoding `latin-1` (com o padrão UTF-8 os acentos dão erro). Leio tudo como texto (`dtype=str`) para o pandas não mudar nada sem eu pedir; as conversões faço depois.

In [ ]:
import pandas as pd

df = pd.read_csv("ocorrencia.csv", sep=";", encoding="latin-1", dtype=str)

In [ ]:
# Quantidade de linhas (ocorrências) e colunas (atributos)
print("Registros:", df.shape[0])
print("Atributos:", df.shape[1])

Registros: 14626
Atributos: 22


In [ ]:
# Primeiras linhas
df.head()

,codigo_ocorrencia,codigo_ocorrencia1,codigo_ocorrencia2,codigo_ocorrencia3,codigo_ocorrencia4,ocorrencia_classificacao,ocorrencia_latitude,ocorrencia_longitude,ocorrencia_cidade,ocorrencia_uf,...,ocorrencia_dia,ocorrencia_hora,investigacao_aeronave_liberada,investigacao_status,divulgacao_relatorio_numero,divulgacao_relatorio_publicado,divulgacao_dia_publicacao,total_recomendacoes,total_aeronaves_envolvidas,ocorrencia_saida_pista
0,88808,88808,88808,88808,88808,INCIDENTE,-23.43555555555,-46.47305555555,GUARULHOS,SP,...,19/04/2026,04:00:00,SIM,FINALIZADA,***,NÃO,NaN,0,1,NÃO
1,88804,88804,88804,88804,88804,INCIDENTE,-23.43555555555,-46.47305555555,GUARULHOS,SP,...,23/04/2026,15:00:00,SIM,FINALIZADA,***,NÃO,NaN,0,1,NÃO
2,88794,88794,88794,88794,88794,INCIDENTE,-22.81,-43.25055555555,RIO DE JANEIRO,RJ,...,28/04/2026,18:36:00,SIM,FINALIZADA,***,NÃO,NaN,0,1,NÃO
3,88783,88783,88783,88783,88783,INCIDENTE,-22.81,-43.25055555555,RIO DE JANEIRO,RJ,...,30/04/2026,19:30:00,SIM,FINALIZADA,***,NÃO,NaN,0,1,NÃO
4,88769,88769,88769,88769,88769,INCIDENTE GRAVE,-12.90861111111,-38.3225,SALVADOR,BA,...,09/05/2026,19:15:00,SIM,ATIVA,A DEFINIR,NÃO,NaN,0,1,SIM


In [ ]:
# Tipos de dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14626 entries, 0 to 14625
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   codigo_ocorrencia               14626 non-null  object
 1   codigo_ocorrencia1              14626 non-null  object
 2   codigo_ocorrencia2              14626 non-null  object
 3   codigo_ocorrencia3              14626 non-null  object
 4   codigo_ocorrencia4              14626 non-null  object
 5   ocorrencia_classificacao        14626 non-null  object
 6   ocorrencia_latitude             11685 non-null  object
 7   ocorrencia_longitude            11685 non-null  object
 8   ocorrencia_cidade               14626 non-null  object
 9   ocorrencia_uf                   14626 non-null  object
 10  ocorrencia_pais                 14626 non-null  object
 11  ocorrencia_aerodromo            14626 non-null  object
 12  ocorrencia_dia                  14626 non-null

Todas as colunas são texto (`str`), porque pedi `dtype=str`. Vou converter datas e números mais adiante.

**O que percebi olhando as colunas:**
* Existem 5 colunas com nome parecido (`codigo_ocorrencia`, `codigo_ocorrencia1`... `codigo_ocorrencia4`), que parecem repetidas.
* Latitude e longitude vêm como texto.
* `ocorrencia_pais` parece ter sempre o mesmo valor.
* Há textos como `***` e `NULL`.

### 2.2 Colunas repetidas ou quase sem variação
Primeiro, será que as 5 colunas de código são iguais?

In [ ]:
for coluna in ["codigo_ocorrencia1", "codigo_ocorrencia2", "codigo_ocorrencia3", "codigo_ocorrencia4"]:
    iguais = (df["codigo_ocorrencia"] == df[coluna]).all()
    print(coluna, "igual a codigo_ocorrencia?", iguais)

codigo_ocorrencia1 igual a codigo_ocorrencia? True
codigo_ocorrencia2 igual a codigo_ocorrencia? True
codigo_ocorrencia3 igual a codigo_ocorrencia? True
codigo_ocorrencia4 igual a codigo_ocorrencia? True


In [ ]:
# Quantos valores diferentes existem em cada coluna
df.nunique().sort_values()

,0
ocorrencia_pais,1
investigacao_status,2
investigacao_aeronave_liberada,2
divulgacao_relatorio_publicado,2
ocorrencia_saida_pista,2
ocorrencia_classificacao,3
total_aeronaves_envolvidas,5
total_recomendacoes,24
ocorrencia_uf,28
divulgacao_dia_publicacao,388


In [ ]:
# Valores das colunas de poucas categorias
for coluna in ["ocorrencia_classificacao", "ocorrencia_pais", "investigacao_aeronave_liberada",
               "investigacao_status", "divulgacao_relatorio_publicado", "ocorrencia_saida_pista",
               "total_aeronaves_envolvidas"]:
    print("=====", coluna, "=====")
    print(df[coluna].value_counts(dropna=False))
    print()

===== ocorrencia_classificacao =====
ocorrencia_classificacao
INCIDENTE          10363
ACIDENTE            3003
INCIDENTE GRAVE     1260
Name: count, dtype: int64

===== ocorrencia_pais =====
ocorrencia_pais
BRASIL    14626
Name: count, dtype: int64

===== investigacao_aeronave_liberada =====
investigacao_aeronave_liberada
SIM    14577
NÃO       49
Name: count, dtype: int64

===== investigacao_status =====
investigacao_status
FINALIZADA    14074
ATIVA           552
Name: count, dtype: int64

===== divulgacao_relatorio_publicado =====
divulgacao_relatorio_publicado
NÃO    12026
SIM     2600
Name: count, dtype: int64

===== ocorrencia_saida_pista =====
ocorrencia_saida_pista
NÃO    13705
SIM      921
Name: count, dtype: int64

===== total_aeronaves_envolvidas =====
total_aeronaves_envolvidas
1    14436
2      183
3        5
0        1
4        1
Name: count, dtype: int64



Conclusões:
* As 5 colunas de código são idênticas: só preciso de uma.
* `ocorrencia_pais` é sempre `BRASIL`, e `investigacao_aeronave_liberada` é `SIM` em quase tudo (só 49 `NÃO`): não ajudam a comparar as ocorrências.

### 2.3 Valores ausentes
O pandas já trata células vazias e o texto `NULL` como ausentes (`NaN`). Vejo quantos existem por coluna:

In [ ]:
df.isna().sum()

,0
codigo_ocorrencia,0
codigo_ocorrencia1,0
codigo_ocorrencia2,0
codigo_ocorrencia3,0
codigo_ocorrencia4,0
ocorrencia_classificacao,0
ocorrencia_latitude,2941
ocorrencia_longitude,2941
ocorrencia_cidade,0
ocorrencia_uf,0


Além dos ausentes "de verdade", a fonte usa `***` e `**` quando não tem a informação. Conto quantas vezes aparecem:

In [ ]:
marcadores = ["***", "**"]

for coluna in df.columns:
    qtd = df[coluna].isin(marcadores).sum()
    if qtd > 0:
        print(coluna, "->", qtd, "vezes")

ocorrencia_latitude -> 1261 vezes
ocorrencia_longitude -> 1262 vezes
ocorrencia_uf -> 25 vezes
divulgacao_relatorio_numero -> 10506 vezes


Destaques:
* `ocorrencia_latitude` e `ocorrencia_longitude`: cerca de 4 mil linhas sem informação (vazio ou `***`) Também vi que uma parte usa **vírgula** no lugar do ponto decimal (ex.: `-14,92416`).
* `divulgacao_dia_publicacao`: 12 mil linhas ausentes, porque a maioria das ocorrências não tem relatório publicado.
* `divulgacao_relatorio_numero`: mistura de números de relatório com `***` e `A DEFINIR`.
* `ocorrencia_uf`: 25 linhas com `***`.
* `ocorrencia_hora`: 2 linhas ausentes.

### 2.4 Duplicidades
Como cada ocorrência tem um código, verifico se algum código aparece mais de uma vez:

In [ ]:
print("Códigos repetidos:", df["codigo_ocorrencia"].duplicated().sum())
print("Linhas totalmente duplicadas:", df.duplicated().sum())

Códigos repetidos: 0
Linhas totalmente duplicadas: 0


## 3. Preparação dos dados

### 3.1 Remover colunas que não serão usadas
* `codigo_ocorrencia1` a `codigo_ocorrencia4`: idênticas a `codigo_ocorrencia`.
* `ocorrencia_pais` e `investigacao_aeronave_liberada`: quase sem variação.
* `ocorrencia_latitude` e `ocorrencia_longitude`: cerca de 29% sem informação, usam vírgula e ponto ao mesmo tempo, e não são necessárias para a análise deste trabalho.
* `divulgacao_relatorio_numero` e `divulgacao_dia_publicacao`: não uso o número nem a data do relatório; para saber se há relatório, uso `divulgacao_relatorio_publicado`.

In [ ]:
colunas_para_remover = [
    "codigo_ocorrencia1", "codigo_ocorrencia2", "codigo_ocorrencia3", "codigo_ocorrencia4",
    "ocorrencia_pais", "investigacao_aeronave_liberada",
    "ocorrencia_latitude", "ocorrencia_longitude",
    "divulgacao_relatorio_numero", "divulgacao_dia_publicacao",
]
df = df.drop(columns=colunas_para_remover)
print("Colunas restantes:", df.shape[1])

Colunas restantes: 12


### 3.2 Renomear colunas
Nomes mais curtos deixam o código mais fácil de escrever e de ler.

In [ ]:
df = df.rename(columns={
    "codigo_ocorrencia": "codigo",
    "ocorrencia_classificacao": "classificacao",
    "ocorrencia_cidade": "cidade",
    "ocorrencia_uf": "uf",
    "ocorrencia_aerodromo": "aerodromo",
    "ocorrencia_dia": "data",
    "ocorrencia_hora": "hora",
    "investigacao_status": "status_investigacao",
    "divulgacao_relatorio_publicado": "relatorio_publicado",
    "total_recomendacoes": "recomendacoes",
    "total_aeronaves_envolvidas": "aeronaves",
    "ocorrencia_saida_pista": "saida_pista",
})
df.columns

Index(['codigo', 'classificacao', 'cidade', 'uf', 'aerodromo', 'data', 'hora',
       'status_investigacao', 'relatorio_publicado', 'recomendacoes',
       'aeronaves', 'saida_pista'],
      dtype='object')

### 3.3 Trocar os marcadores `***` por ausente
Assim o pandas passa a tratar esses casos como "sem informação". **Não vou preencher** os ausentes com outro valor (por exemplo, a UF mais comum), porque isso inventaria informação. São poucos casos (25 linhas na UF); nas análises, essas linhas simplesmente ficam de fora.

In [ ]:
df = df.replace(marcadores, pd.NA)
print("UF ausente:", df["uf"].isna().sum())

UF ausente: 25


### 3.4 Conversões de tipo e colunas novas
* **Data:** de texto (`19/04/2026`) para data (`datetime`), para poder pegar o ano.
* **Hora:** de `04:00:00` para o número da hora (`4`).
* **Números:** `recomendacoes` e `aeronaves` de texto para número inteiro.
* **Sim/Não:** `saida_pista` e `relatorio_publicado` viram `True`/`False`. Assim, a média dessa coluna dá direto a porcentagem de "SIM".

In [ ]:
# Data e ano
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year

# Hora: fico só com os 2 primeiros caracteres ("04:00:00" -> "04" -> 4)
df["hora"] = pd.to_numeric(df["hora"].str[:2])

# Números
df["recomendacoes"] = df["recomendacoes"].astype(int)
df["aeronaves"] = df["aeronaves"].astype(int)

# SIM / NÃO -> True / False
df["saida_pista"] = df["saida_pista"] == "SIM"
df["relatorio_publicado"] = df["relatorio_publicado"] == "SIM"

df.dtypes

,0
codigo,object
classificacao,object
cidade,object
uf,object
aerodromo,object
data,datetime64[ns]
hora,float64
status_investigacao,object
relatorio_publicado,bool
recomendacoes,int64


### 3.5 Período do dia
Crio uma função que recebe a hora e devolve o período do dia. Depois aplico a função em toda a coluna com `apply`.

In [ ]:
def periodo_do_dia(hora):
    """Recebe a hora (0 a 23) e devolve o período do dia."""
    if pd.isna(hora):
        return None
    elif hora < 6:
        return "madrugada"
    elif hora < 12:
        return "manhã"
    elif hora < 18:
        return "tarde"
    else:
        return "noite"

df["periodo"] = df["hora"].apply(periodo_do_dia)
df[["hora", "periodo"]].head()

,hora,periodo
0,4.0,madrugada
1,15.0,tarde
2,18.0,noite
3,19.0,noite
4,19.0,noite


### 3.6 Conferindo o resultado

In [ ]:
print("Primeira data:", df["data"].min().date())
print("Última data:", df["data"].max().date())
print()
print("Hora mínima:", df["hora"].min(), "| máxima:", df["hora"].max())
print()
df.isna().sum()

Primeira data: 2007-01-01
Última data: 2026-05-09

Hora mínima: 0.0 | máxima: 23.0



,0
codigo,0
classificacao,0
cidade,0
uf,25
aerodromo,0
data,0
hora,2
status_investigacao,0
relatorio_publicado,0
recomendacoes,0


As datas vão de 2007 a maio de 2026 (2026 está incompleto, o que preciso lembrar ao comparar anos), e as horas estão entre 0 e 23, como esperado.

### 3.7 Resumo das decisões
Basicamente tinha 5 colunas iguais, eu mantive só uma. Removi país e aeronave liberada, já que não tinham varaiação, removi latitude e longitude por causa da augência de dados, removi número de data e hora pois não pretendo usar na análise. troquei *** e ** por ausente, converti data e hora e mudei "SIM" e "NÃO" para "TRUE" e "FALSE", para facilitar cálculo.

In [ ]:
# Salva a base tratada
df.to_csv("ocorrencia_tratado.csv", index=False, encoding="utf-8")
print("Base tratada:", df.shape[0], "linhas e", df.shape[1], "colunas")

Base tratada: 14626 linhas e 14 colunas


## 4. Análise dos dados
Primeiro, uma função que vou reutilizar: ela mostra as categorias mais frequentes de uma coluna, com a quantidade e a porcentagem (sobre as linhas que têm informação).

In [ ]:
def mais_frequentes(dados, coluna, n=10):
    """Mostra as n categorias mais comuns de uma coluna, com quantidade e %."""
    contagem = dados[coluna].value_counts().head(n)
    total_validos = dados[coluna].notna().sum()
    tabela = pd.DataFrame({
        "ocorrencias": contagem,
        "percentual": (contagem / total_validos * 100).round(1),
    })
    return tabela

### 4.1 Visão geral e classificação

In [ ]:
print("Total de ocorrências:", len(df))
print("Anos cobertos:", df["ano"].min(), "a", df["ano"].max())
print("Ocorrências com mais de uma aeronave:", (df["aeronaves"] > 1).sum())

Total de ocorrências: 14626
Anos cobertos: 2007 a 2026
Ocorrências com mais de uma aeronave: 189


In [ ]:
mais_frequentes(df, "classificacao")

,ocorrencias,percentual
classificacao,,
INCIDENTE,10363,70.9
ACIDENTE,3003,20.5
INCIDENTE GRAVE,1260,8.6


### 4.2 Evolução por ano
`crosstab` conta quantas ocorrências existem em cada combinação de ano e classificação. Lembrando: 2026 está incompleto.

In [ ]:
por_ano = pd.crosstab(df["ano"], df["classificacao"], margins=True, margins_name="TOTAL")
por_ano

classificacao,ACIDENTE,INCIDENTE,INCIDENTE GRAVE,TOTAL
ano,,,,
2007,103,432,21,556
2008,108,310,38,456
2009,113,347,39,499
2010,128,363,61,552
2011,187,324,65,576
2012,205,365,77,647
2013,199,386,69,654
2014,175,318,76,569
2015,172,250,49,471


Algo interessante é que os acidentes ficam parecidos de um ano para outro, mas os incidentes aumentam muito a partir de 2023. Vou ver isso melhor comparando dois períodos:

In [ ]:
# Média anual de ocorrências antes e depois de 2023 (sem 2026, que está incompleto)
antes = df[(df["ano"] >= 2019) & (df["ano"] <= 2022)]
depois = df[(df["ano"] >= 2023) & (df["ano"] <= 2025)]

media_antes = antes["classificacao"].value_counts() / 4
media_depois = depois["classificacao"].value_counts() / 3

comparacao = pd.DataFrame({"media_2019_2022": media_antes, "media_2023_2025": media_depois})
comparacao.round(0)

,media_2019_2022,media_2023_2025
classificacao,,
INCIDENTE,318.0,1647.0
ACIDENTE,145.0,161.0
INCIDENTE GRAVE,81.0,72.0


### 4.3 Onde acontecem: UF, cidade e aeródromo
São contagens absolutas: estados e cidades com mais voos naturalmente têm mais ocorrências. O dataset não traz o número de voos, então não dá para calcular uma taxa.

In [ ]:
mais_frequentes(df, "uf")

,ocorrencias,percentual
uf,,
SP,3626,24.8
RJ,1526,10.5
MG,1224,8.4
PR,988,6.8
RS,776,5.3
MT,665,4.6
GO,664,4.5
BA,655,4.5
PA,621,4.3


In [ ]:
mais_frequentes(df, "cidade")

,ocorrencias,percentual
cidade,,
RIO DE JANEIRO,1088,7.4
GUARULHOS,857,5.9
SÃO PAULO,786,5.4
CAMPINAS,564,3.9
BRASÍLIA,472,3.2
SALVADOR,373,2.6
RECIFE,347,2.4
GOIÂNIA,340,2.3
BELO HORIZONTE,333,2.3


In [ ]:
mais_frequentes(df, "aerodromo")

,ocorrencias,percentual
aerodromo,,
FAER,4069,27.8
SBGR,721,4.9
SBGL,452,3.1
SBKP,422,2.9
SBBR,356,2.4
SBSP,355,2.4
SBSV,296,2.0
SBRF,294,2.0
SBPA,270,1.8


Agora, o mesmo `for` para ver os 5 estados com mais ocorrências em cada classificação:

In [ ]:
for tipo in ["ACIDENTE", "INCIDENTE GRAVE", "INCIDENTE"]:
    filtro = df[df["classificacao"] == tipo]
    print("=====", tipo, "-", len(filtro), "ocorrências =====")
    print(mais_frequentes(filtro, "uf", 5))
    print()

===== ACIDENTE - 3003 ocorrências =====
    ocorrencias  percentual
uf                         
SP          595        19.8
MT          323        10.8
RS          259         8.6
MG          238         7.9
PA          211         7.0

===== INCIDENTE GRAVE - 1260 ocorrências =====
    ocorrencias  percentual
uf                         
SP          217        17.3
GO          125         9.9
MG          115         9.1
PR           92         7.3
MT           92         7.3

===== INCIDENTE - 10363 ocorrências =====
    ocorrencias  percentual
uf                         
SP         2814        27.2
RJ         1336        12.9
MG          871         8.4
PR          686         6.6
BA          477         4.6



### 4.4 Horário do dia

In [ ]:
# Ocorrências por hora do dia
df["hora"].value_counts().sort_index()

,count
hora,
0.0,515
1.0,236
2.0,214
3.0,236
4.0,214
5.0,158
6.0,115
7.0,108
8.0,180


In [ ]:
# Período do dia
mais_frequentes(df, "periodo")

,ocorrencias,percentual
periodo,,
tarde,5959,40.7
noite,4727,32.3
manhã,2365,16.2
madrugada,1573,10.8


In [ ]:
# Período do dia dentro de cada classificação (cada linha soma 100%)
(pd.crosstab(df["classificacao"], df["periodo"], normalize="index") * 100).round(1)

periodo,madrugada,manhã,noite,tarde
classificacao,,,,
ACIDENTE,1.3,15.9,36.8,46.0
INCIDENTE,14.5,16.8,30.7,38.0
INCIDENTE GRAVE,2.5,12.0,34.7,50.8


### 4.5 Saída de pista
Como `saida_pista` é `True`/`False`, a média dá a proporção de ocorrências com saída de pista.

In [ ]:
saida = df.groupby("classificacao")["saida_pista"].agg(["count", "sum", "mean"])
saida.columns = ["ocorrencias", "com_saida_de_pista", "percentual"]
saida["percentual"] = (saida["percentual"] * 100).round(1)
saida

,ocorrencias,com_saida_de_pista,percentual
classificacao,,,
ACIDENTE,3003,454,15.1
INCIDENTE,10363,120,1.2
INCIDENTE GRAVE,1260,347,27.5


### 4.6 Investigação, relatório e recomendações

In [ ]:
# Situação da investigação por classificação
pd.crosstab(df["classificacao"], df["status_investigacao"], margins=True)

status_investigacao,ATIVA,FINALIZADA,All
classificacao,,,
ACIDENTE,466,2537,3003
INCIDENTE,10,10353,10363
INCIDENTE GRAVE,76,1184,1260
All,552,14074,14626


In [ ]:
# Relatório publicado (%) e recomendações de segurança por classificação
resumo = df.groupby("classificacao").agg(
    ocorrencias=("codigo", "count"),
    relatorio_publicado=("relatorio_publicado", "mean"),
    recomendacoes_total=("recomendacoes", "sum"),
    recomendacoes_media=("recomendacoes", "mean"),
)
resumo["relatorio_publicado"] = (resumo["relatorio_publicado"] * 100).round(1)
resumo["recomendacoes_media"] = resumo["recomendacoes_media"].round(2)
resumo

,ocorrencias,relatorio_publicado,recomendacoes_total,recomendacoes_media
classificacao,,,,
ACIDENTE,3003,68.2,2946,0.98
INCIDENTE,10363,0.3,99,0.01
INCIDENTE GRAVE,1260,41.7,642,0.51


### 4.7 Ocorrências com mais de uma aeronave

In [ ]:
mais_frequentes(df, "aeronaves")

,ocorrencias,percentual
aeronaves,,
1,14436,98.7
2,183,1.3
3,5,0.0
0,1,0.0
4,1,0.0


In [ ]:
# Só as ocorrências com 2 ou mais aeronaves: qual a classificação delas?
varias = df[df["aeronaves"] >= 2]
mais_frequentes(varias, "classificacao")

,ocorrencias,percentual
classificacao,,
INCIDENTE,133,70.4
INCIDENTE GRAVE,41,21.7
ACIDENTE,15,7.9


## 5. Documentação e conclusões

### 5.1 Como reproduzir
* Requisito: Python 3 com pandas.
* Coloque `ocorrencia.csv` na mesma pasta do notebook e execute as células em ordem (Kernel > Restart & Run All).
* O notebook segue as etapas: 1) problema, 2) exploração, 3) preparação (resumo em 3.7), 4) análise, 5) conclusões.

### 5.2 Conclusões
O arquivo tem 14.626 ocorrências (de janeiro de 2007 a maio de 2026), sendo 70,9% incidentes, 20,5% acidentes e 8,6% incidentes graves.

1. **Evolução no tempo.** O número de acidentes é estável: em média 145 por ano em 2019–2022 e 161 por ano em 2023–2025. Já os incidentes dispararam a partir de 2023: a média foi de 318 por ano em 2019–2022 para 1.647 por ano em 2023–2025 (cerca de 5 vezes mais), com pico em 2024 (2.456 incidentes). Como os acidentes não acompanharam esse aumento, o mais provável é que o crescimento venha de mudanças na quantidade ou na forma de notificação, mas essa é uma hipótese que eu não testei neste trabalho.
2. **Onde.** São Paulo lidera (24,8% das ocorrências), seguido de Rio de Janeiro (10,5%) e Minas Gerais (8,4%); esses três, mais Paraná e Rio Grande do Sul, somam 55,8%. Entre as cidades, aparecem Rio de Janeiro, Guarulhos e São Paulo. Nos acidentes, o ranking muda: SP (19,8%), Mato Grosso (10,8%) e RS (8,6%). Como o dataset não traz o número de voos, esses números são contagens e não taxas.
3. **Horário.** As ocorrências se concentram à tarde (40,7%) e à noite (32,3%), com picos perto das 13h e das 20h. A madrugada pesa muito pouco nos acidentes (1,3%), mas bem mais nos incidentes (14,5%).
4. **Saída de pista.** Acontece em 27,5% dos incidentes graves, 15,1% dos acidentes e só 1,2% dos incidentes.
5. **Investigação.** Quase todas as ocorrências já estão finalizadas (96,2%); as 552 ainda ativas são principalmente acidentes (466). Os acidentes têm relatório publicado em cerca de 68% dos casos e geram em média 0,98 recomendação de segurança cada, contra 0,51 nos incidentes graves e 0,01 nos incidentes.
6. **Mais de uma aeronave.** Só 1,3% das ocorrências envolvem 2 ou mais aeronaves (189 casos), e a maioria delas (70%) é incidente.


### 5.3 Limitações
* Cada linha é uma ocorrência notificada ao CENIPA. A quantidade depende também de como e quanto se notifica, e não só de quantas ocorrências realmente aconteceram.
* 2026 está incompleto (até maio), então não deve ser comparado com anos fechados.
* Os dados não trazem o número de voos, então comparar estados ou aeroportos só por quantidade de ocorrências pode enganar (quem tem mais voos tem mais ocorrências).
* A análise é descritiva: mostra o que aconteceu, mas não prova as causas.

### 5.4 Ideias para melhorar depois
* Usar as **tabelas de aeronave e de fator contribuinte** do mesmo conjunto de dados (juntando com `merge` pelo código da ocorrência) para ver tipos de aeronave e causas.
* Fazer **gráficos** (evolução por ano, ocorrências por hora) quando você aprender matplotlib.
* Usar **latitude e longitude** para um mapa das ocorrências.
* Investigar por que os incidentes aumentaram a partir de 2023.